In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import (
    collect_set,
    size,
    col,
    array_contains,
    array_except
)

# Create Purchase DataFrame

In [0]:

def create_purchase_df():

    schema = StructType([
        StructField("customer", IntegerType(), True),
        StructField("product_model", StringType(), True)
    ])

    data = [
        (1, "iphone13"),
        (1, "dell i5 core"),
        (2, "iphone13"),
        (2, "dell i5 core"),
        (3, "iphone13"),
        (3, "dell i5 core"),
        (1, "dell i3 core"),
        (1, "hp i5 core"),
        (1, "iphone14"),
        (3, "iphone14"),
        (4, "iphone13")
    ]

    return spark.createDataFrame(data, schema)

# Create Product DataFrame

In [0]:

def create_product_df():

    schema = StructType([
        StructField("product_model", StringType(), True)
    ])

    data = [
        ("iphone13",),
        ("dell i5 core",),
        ("dell i3 core",),
        ("hp i5 core",),
        ("iphone14",)
    ]

    return spark.createDataFrame(data, schema)


# Question 2
# Customers who bought only iphone13

In [0]:

def customers_only_iphone13(df):

    return (
        df.groupBy("customer")
        .agg(
            collect_set("product_model").alias("products")
        )
        .filter(
            (size(col("products")) == 1) &
            (col("products")[0] == "iphone13")
        )
    )


# Question 3
# Customers upgraded

In [0]:

def upgraded_customers(df):

    return (
        df.groupBy("customer")
        .agg(
            collect_set("product_model").alias("products")
        )
        .filter(
            array_contains(col("products"), "iphone13") &
            array_contains(col("products"), "iphone14")
        )
    )

# Question 4
# Customers bought every product

In [0]:
def customers_all_products(purchase_df, product_df):

    all_products = set(
        row.product_model
        for row in product_df.collect()
    )

    customer_products = (
        purchase_df
        .groupBy("customer")
        .agg(
            collect_set("product_model").alias("products")
        )
        .collect()
    )

    result = []

    for row in customer_products:
        if set(row.products) == all_products:
            result.append((row.customer,))

    return spark.createDataFrame(
        result,
        ["customer"]
    )